# 1. Commits

A commit is a snapshot of the whole project, plus a message saying why. By
the end of this chapter the history looks like this:

```mermaid
gitGraph
    commit id: "volume + displacement"
    commit id: "block coefficient"
    commit id: "checks"
    commit id: "buoyancy"
    commit id: "ignore caches"
```

In [1]:
import os, shutil, stat, sys
from pathlib import Path


def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)  # git makes its object files read-only
    func(path)


work = Path.home() / "git-course" / "01-commits"
if work.exists():
    shutil.rmtree(work, onexc=remove_readonly)
work.mkdir(parents=True)
%cd {work}

/home/emircem/git-course/01-commits


## A repository

`git init` turns the current folder into a repository. It creates one
hidden folder, `.git`, where the history is kept.

In [2]:
!git init

Initialized empty Git repository in /home/emircem/git-course/01-commits/.git/


In [3]:
os.listdir()

['.git']

In [4]:
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


No commits yet, nothing to commit. Let's write some code.

In [5]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Writing hydrostatics.py


In [6]:
!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	hydrostatics.py

nothing added to commit but untracked files present (use "git add" to track)


The file is **untracked**: git sees it but is not saving it. Git never
saves anything until you tell it to. First you **add** a file to the
staging area — the list of things that go into the next commit:

In [7]:
!git add hydrostatics.py

In [8]:
!git status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   hydrostatics.py



Now it is staged. **Commit** saves the snapshot:

In [9]:
!git commit -m "Add volume and displacement for a box barge"

[main (root-commit) 94f3e0a] Add volume and displacement for a box barge
 1 file changed, 13 insertions(+)
 create mode 100644 hydrostatics.py


In [10]:
!git log

commit 94f3e0a2397d7fdb1222e061e07b310857f13d10 (HEAD -> main)
Author: Emir Cem Gezer <emir@company.com>
Date:   Mon Oct 19 23:42:03 2026 +0200

    Add volume and displacement for a box barge


## Changing the file

Real hulls are not boxes. The **block coefficient** `cb` is the fraction
of the box the hull really fills: 0.85 for a tanker, 0.5 for a patrol boat.

In [11]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft, cb=1.0):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft * cb


def displacement(length, beam, draft, cb=1.0):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft, cb)

Overwriting hydrostatics.py


`git diff` shows what changed since the last commit. Lines starting with
`-` were removed, lines starting with `+` were added.

In [12]:
!git diff

diff --git a/hydrostatics.py b/hydrostatics.py
index f3c3cd6..f18d6a0 100644
--- a/hydrostatics.py
+++ b/hydrostatics.py
@@ -3,11 +3,11 @@
 RHO = 1025.0  # kg/m^3, sea water
 
 
-def volume(length, beam, draft):
+def volume(length, beam, draft, cb=1.0):
     """Immersed volume of the hull, in m^3."""
-    return length * beam * draft
+    return length * beam * draft * cb
 
 
-def displacement(length, beam, draft):
+def displacement(length, beam, draft, cb=1.0):
     """Mass of the water the hull pushes aside, in kg."""
-    return RHO * volume(length, beam, draft)
+    return RHO * volume(length, beam, draft, cb)


In [13]:
!git add hydrostatics.py
!git commit -m "Scale volume by the block coefficient"

[main 3aaf271] Scale volume by the block coefficient
 1 file changed, 4 insertions(+), 4 deletions(-)


A good message says **why**, in a short line: "Add …", "Fix …", "Scale …".
Messages like `update` or `stuff` are no help to anyone later — including
you.

## A third commit

In [14]:
%%writefile test_hydrostatics.py
from hydrostatics import displacement, volume

assert volume(10, 4, 2) == 80
assert volume(10, 4, 2, 0.5) == 40
assert displacement(10, 4, 2) == 82000  # 80 m^3 of sea water

print("all checks passed")

Writing test_hydrostatics.py


In [15]:
!"{sys.executable}" test_hydrostatics.py

all checks passed


In [16]:
!git add test_hydrostatics.py
!git commit -m "Add checks for volume and displacement"

[main 2f7ed27] Add checks for volume and displacement
 1 file changed, 7 insertions(+)
 create mode 100644 test_hydrostatics.py


In [17]:
!git log --oneline

2f7ed27 (HEAD -> main) Add checks for volume and displacement
3aaf271 Scale volume by the block coefficient
94f3e0a Add volume and displacement for a box barge


`--oneline` is the short view you will use every day. **HEAD** marks the
commit you are on now.

## Fixing the last commit

Buoyancy is the force the water pushes back with. Let's add it, and commit
it with a bad message on purpose.

In [18]:
%%writefile -a hydrostatics.py


def buoyancy(length, beam, draft, cb=1.0):
    """Upward force on the hull, in N."""
    return 9.81 * displacement(length, beam, draft, cb)

Appending to hydrostatics.py


In [19]:
!git add hydrostatics.py
!git commit -m "stuff"

[main d482446] stuff
 1 file changed, 5 insertions(+)


`--amend` replaces the last commit with a corrected one:

In [20]:
!git commit --amend -m "Add buoyancy force from displacement"

[main b9f1cd4] Add buoyancy force from displacement
 Date: Mon Oct 19 23:42:05 2026 +0200
 1 file changed, 5 insertions(+)


In [21]:
!git log --oneline

b9f1cd4 (HEAD -> main) Add buoyancy force from displacement
2f7ed27 Add checks for volume and displacement
3aaf271 Scale volume by the block coefficient
94f3e0a Add volume and displacement for a box barge


Only amend commits you have not shared yet (chapter 5 explains why).

`git show` prints one commit and what it changed:

In [22]:
!git show HEAD

commit b9f1cd41b9eb3e2ab02c126fc22b945e9d6cc510 (HEAD -> main)
Author: Emir Cem Gezer <emir@company.com>
Date:   Mon Oct 19 23:42:05 2026 +0200

    Add buoyancy force from displacement

diff --git a/hydrostatics.py b/hydrostatics.py
index f18d6a0..c2c104f 100644
--- a/hydrostatics.py
+++ b/hydrostatics.py
@@ -11,3 +11,8 @@ def volume(length, beam, draft, cb=1.0):
 def displacement(length, beam, draft, cb=1.0):
     """Mass of the water the hull pushes aside, in kg."""
     return RHO * volume(length, beam, draft, cb)
+
+
+def buoyancy(length, beam, draft, cb=1.0):
+    """Upward force on the hull, in N."""
+    return 9.81 * displacement(length, beam, draft, cb)


## Throwing away a change

Someone adds nonsense to the file:

In [23]:
%%writefile -a hydrostatics.py

RHO = 1.2  # air?!

Appending to hydrostatics.py


In [24]:
# This one fails on purpose: the checks catch the mistake.
!"{sys.executable}" test_hydrostatics.py

Traceback (most recent call last):
  File "/home/emircem/git-course/01-commits/test_hydrostatics.py", line 5, in <module>
    assert displacement(10, 4, 2) == 82000  # 80 m^3 of sea water
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError


`git restore` puts the file back to how it was in the last commit:

In [25]:
!git restore hydrostatics.py

In [26]:
!"{sys.executable}" test_hydrostatics.py

all checks passed


## Un-staging a file

A file with a password must never be committed. If you add it by mistake,
`git restore --staged` takes it back out of the staging area:

In [27]:
%%writefile credentials.txt
secret-password-do-not-commit

Writing credentials.txt


In [28]:
!git add credentials.txt
!git status

On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   credentials.txt

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/



In [29]:
!git restore --staged credentials.txt
!git status

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/
	credentials.txt

nothing added to commit but untracked files present (use "git add" to track)


## .gitignore

A `.gitignore` file lists files git should never track — secrets, and the
`__pycache__` folder Python creates.

In [30]:
%%writefile .gitignore
__pycache__/
credentials.txt

Writing .gitignore


In [31]:
!git status

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

nothing added to commit but untracked files present (use "git add" to track)


In [32]:
!git add .gitignore
!git commit -m "Ignore caches and local credentials"

[main 9af33a4] Ignore caches and local credentials
 1 file changed, 2 insertions(+)
 create mode 100644 .gitignore


In [33]:
!git log --oneline

9af33a4 (HEAD -> main) Ignore caches and local credentials
b9f1cd4 Add buoyancy force from displacement
2f7ed27 Add checks for volume and displacement
3aaf271 Scale volume by the block coefficient
94f3e0a Add volume and displacement for a box barge


## Summary

| I want to… | Command |
| - | - |
| start a repository | `git init` |
| see what is going on | `git status` |
| stage a file | `git add <file>` |
| save a snapshot | `git commit -m "message"` |
| see what changed | `git diff` |
| see the history | `git log --oneline` |
| fix the last commit | `git commit --amend -m "message"` |
| throw away a change | `git restore <file>` |
| un-stage a file | `git restore --staged <file>` |

## Exercises

1. Add a `waterplane_area(length, beam)` function and commit it with a good
   message.
2. Try `git log --stat` and `git log -p`. What do they add?

Next: [2. Branches](02-branches.ipynb)